---
# <font color="#CA3532">Deep Learning Fundamentals and Basic Tools (2026/2027) - Final project</font>
---


In this project you will implement and compare four deep-learning architectures for **multiclass human activity recognition**:

- Deep Neural Network (DNN)
- Vanilla Recurrent Neural Network (RNN)
- 1D Convolutional Neural Network (CNN)
- Long Short-Term Memory Network (LSTM)

You will work with the **UCI Human Activity Recognition Using Smartphones (UCI HAR)** dataset.

You must compare predictive performance, class-level errors, convergence, model complexity, training time, inference time, and stability across repeated runs.

# Generative-AI Policy

This project must represent **your own programming, experimentation, analysis, and understanding**.

## Generative AI is prohibited

The use of any generative-AI or Large Language Model (LLM) to generate, complete, rewrite, or substantially assist with the code, experiments, analysis, or written answers for this project is **not permitted**.

**Work generated using LLM assistance will be treated as plagiarism. If it is the case, the corresponding project code will not be evaluated and the student will receive no credit for that work.**

You must write a project cv explaning the following points;

- explain individual functions or model layers;
- explain tensor dimensions;
- explain the training procedure;
- suugest an alternative for part of your implementation;
- justify hyperparameters;
- interpret results, confusion matrices, and all metrics;
- explain your conclusions.

Do not submit code or analysis that you cannot explain.

# 1. Dataset

The dataset that is considered for this project is stored at '"../data/final_projetc"' which is donloaded from UCI repository.

This dataset contains smartphone inertial-sensor recordings collected while participants performed six daily activities.


# 3. Imports

In [ ]:
from pathlib import Path
import json
import sys
import time
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import (
    DNNClassifier,
    RNNClassifier,
    CNNClassifier,
    LSTMClassifier,
)

from src.training import fit

from src.evaluation import (
    evaluate_model,
    count_parameters,
    measure_inference_time,
)

from src.utils import set_seed

print("Imports completed.")

# 4. Prepare the Dataset

The instructor provides `prepare_dataset.py`.

Place the downloaded UCI HAR raw dataset under the location required by the instructor-provided preparation script.

Then run the following cell.

**Do not modify `prepare_dataset.py`.**

It should create:

```text
data/
├── har_multiclass.npz
└── metadata.json
```

The `.npz` file should contain the prepared sequential arrays used throughout the project.

In [ ]:
import subprocess

prepare_script = PROJECT_ROOT / "prepare_dataset.py"
dataset_path = PROJECT_ROOT / "data" / "har_multiclass.npz"
metadata_path = PROJECT_ROOT / "data" / "metadata.json"

if not dataset_path.exists() or not metadata_path.exists():
    print("Prepared dataset not found.")
    print("Running instructor-provided prepare_dataset.py ...")

    subprocess.run(
        [sys.executable, str(prepare_script)],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("Prepared dataset already exists.")

print("Dataset:", dataset_path)
print("Metadata:", metadata_path)

# 5. Load and Inspect Metadata

Read the metadata before loading the arrays.

After running the cell, answer:

1. How many activity classes exist?
2. What is the sequence length?
3. How many sensor channels are used?
4. What are the class names?
5. How many observations belong to each split?
6. How was the validation set constructed?

In [ ]:
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata

### Metadata observations

**TODO:** Write your observations here.

# 6. Load Prepared Sequential Data

In [ ]:
data = np.load(dataset_path)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)

print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)

print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

# 7. Basic Dataset Checks

In [ ]:
assert X_train.ndim == 3
assert X_val.ndim == 3
assert X_test.ndim == 3

assert X_train.shape[1:] == X_val.shape[1:] == X_test.shape[1:]

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

n_classes = len(np.unique(y_train))
sequence_length = X_train.shape[1]
n_features = X_train.shape[2]

print("Classes:", np.unique(y_train))
print("Number of classes:", n_classes)
print("Sequence length:", sequence_length)
print("Sensor channels:", n_features)

# 8. Activity Names and Sensor Channels

In [ ]:
class_names = metadata.get(
    "class_names",
    [
        "WALKING",
        "WALKING_UPSTAIRS",
        "WALKING_DOWNSTAIRS",
        "SITTING",
        "STANDING",
        "LAYING",
    ],
)

channel_names = metadata.get(
    "channel_names",
    [
        "body_acc_x",
        "body_acc_y",
        "body_acc_z",
        "body_gyro_x",
        "body_gyro_y",
        "body_gyro_z",
        "total_acc_x",
        "total_acc_y",
        "total_acc_z",
    ],
)

print("Classes:")
for i, name in enumerate(class_names):
    print(i, name)

print("\nChannels:")
for i, name in enumerate(channel_names):
    print(i, name)

# 9. Class Distribution

Inspect class frequencies in all splits.

In [ ]:
def class_counts(y, n_classes):
    return np.bincount(y, minlength=n_classes)

distribution_df = pd.DataFrame({
    "Class ID": np.arange(n_classes),
    "Activity": class_names,
    "Train": class_counts(y_train, n_classes),
    "Validation": class_counts(y_val, n_classes),
    "Test": class_counts(y_test, n_classes),
})

distribution_df

In [ ]:
plot_df = distribution_df.set_index("Activity")[["Train", "Validation", "Test"]]

plot_df.plot(kind="bar", figsize=(12, 5))

plt.title("Activity Class Distribution")
plt.xlabel("Activity")
plt.ylabel("Number of Windows")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Class-distribution discussion

Discuss whether the activity classes are balanced and why Macro F1 and Weighted F1 should both be reported.

**TODO:** Write your analysis here.

# 10. Visualize Raw Sequential Sensor Data

Visualize one example from every activity.

Begin with the three body-acceleration channels.

In [ ]:
fig, axes = plt.subplots(
    n_classes,
    1,
    figsize=(13, 3 * n_classes),
    sharex=True,
)

for class_id in range(n_classes):
    idx = np.where(y_train == class_id)[0][0]
    sample = X_train[idx]

    axes[class_id].plot(sample[:, 0], label=channel_names[0])
    axes[class_id].plot(sample[:, 1], label=channel_names[1])
    axes[class_id].plot(sample[:, 2], label=channel_names[2])

    axes[class_id].set_title(class_names[class_id])
    axes[class_id].set_ylabel("Signal")
    axes[class_id].legend(loc="upper right")

axes[-1].set_xlabel("Time Step")
plt.tight_layout()
plt.show()

Now inspect the three gyroscope channels.

In [ ]:
fig, axes = plt.subplots(
    n_classes,
    1,
    figsize=(13, 3 * n_classes),
    sharex=True,
)

for class_id in range(n_classes):
    idx = np.where(y_train == class_id)[0][0]
    sample = X_train[idx]

    for channel_idx in [3, 4, 5]:
        axes[class_id].plot(
            sample[:, channel_idx],
            label=channel_names[channel_idx],
        )

    axes[class_id].set_title(class_names[class_id])
    axes[class_id].set_ylabel("Signal")
    axes[class_id].legend(loc="upper right")

axes[-1].set_xlabel("Time Step")
plt.tight_layout()
plt.show()

### Sensor-data observations

Discuss:

- differences between dynamic and static activities;
- similarities between WALKING-related classes;
- similarities between SITTING and STANDING;
- local temporal patterns;
- longer sequential structure;
- why CNN/RNN/LSTM may exploit information differently from a flattened DNN.

**TODO:** Write your observations here.

# 11. Backend Implementation Requirements

You must implement the backend yourself.

Required files:

```text
src/models.py
src/training.py
src/evaluation.py
src/utils.py
```

The notebook should call those implementations. It must not contain replacement implementations for them.

## 11.1 `src/models.py`

Implement:

```python
DNNClassifier
RNNClassifier
CNNClassifier
LSTMClassifier
```

All models receive the same underlying information and must ultimately output:

```text
(batch_size, 6)
```

logits/scores.

### DNN

Transform:

```text
(batch, 128, 9)
```

into an appropriate vector representation.

### RNN

Process the 128 time steps sequentially with a vanilla recurrent architecture.

### CNN

Use one-dimensional convolution over the temporal dimension.

### LSTM

Use an LSTM-based sequence representation for classification.

Important architectural parameters should be configurable.

## 11.2 `src/training.py`

Implement:

```python
fit(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    **kwargs
)
```

It must return at least:

```python
{
    "model": ...,
    "history": {
        "train_loss": ...,
        "val_loss": ...,
        "train_accuracy": ...,
        "val_accuracy": ...
    },
    "training_time": ...,
    "epochs_completed": ...
}
```

Support appropriate training settings including epochs, batch size, learning rate, seed, device, and early stopping.

## 11.3 `src/evaluation.py`

Implement:

```python
evaluate_model(model, X, y, **kwargs)
count_parameters(model)
measure_inference_time(model, X, **kwargs)
```

`evaluate_model` must return at least:

```python
{
    "accuracy": ...,
    "macro_precision": ...,
    "macro_recall": ...,
    "macro_f1": ...,
    "weighted_f1": ...,
    "predictions": ...
}
```

## 11.4 `src/utils.py`

Implement:

```python
set_seed(seed)
```

The function must set the relevant random seeds for reproducibility.

# 12. Instantiate DNN, RNN, CNN and LSTM

You may add additional keyword arguments supported by your implementation.

Document your final architecture choices.

In [ ]:
SEED = 42
set_seed(SEED)

dnn_model = DNNClassifier(
    sequence_length=sequence_length,
    n_features=n_features,
    n_classes=n_classes,
)

rnn_model = RNNClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

cnn_model = CNNClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

lstm_model = LSTMClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

models = {
    "DNN": dnn_model,
    "RNN": rnn_model,
    "CNN": cnn_model,
    "LSTM": lstm_model,
}

for name, model in models.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(model)

# 13. Forward-Pass Sanity Check

All models must return one vector of six class logits for every input sequence.

In [ ]:
sample_batch = torch.tensor(
    X_train[:8],
    dtype=torch.float32,
)

for name, model in models.items():
    model.eval()

    with torch.no_grad():
        output = model(sample_batch)

    print(
        f"{name:5s} | "
        f"input={tuple(sample_batch.shape)} | "
        f"output={tuple(output.shape)}"
    )

    assert output.shape == (8, n_classes)

### Architecture documentation

For each architecture document:

- input representation;
- layer structure;
- hidden/filter dimensions;
- activation functions;
- dropout;
- output representation;
- parameter choices;
- what temporal information the architecture can exploit.

**TODO:** Write your architecture documentation here.

# 14. Training Configuration

Use comparable conditions where reasonable.

Do not use the test set for model selection or hyperparameter tuning.

In [ ]:
TRAINING_CONFIG = {
    "epochs": 50,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "patience": 8,
    "seed": SEED,
}

TRAINING_CONFIG

# 15. Train DNN

In [ ]:
dnn_result = fit(
    dnn_model,
    X_train, y_train,
    X_val, y_val,
    **TRAINING_CONFIG,
)

dnn_model = dnn_result["model"]

print("Training time:", dnn_result["training_time"])
print("Epochs:", dnn_result["epochs_completed"])

# 16. Train RNN

In [ ]:
rnn_result = fit(
    rnn_model,
    X_train, y_train,
    X_val, y_val,
    **TRAINING_CONFIG,
)

rnn_model = rnn_result["model"]

print("Training time:", rnn_result["training_time"])
print("Epochs:", rnn_result["epochs_completed"])

# 17. Train CNN

In [ ]:
cnn_result = fit(
    cnn_model,
    X_train, y_train,
    X_val, y_val,
    **TRAINING_CONFIG,
)

cnn_model = cnn_result["model"]

print("Training time:", cnn_result["training_time"])
print("Epochs:", cnn_result["epochs_completed"])

# 18. Train LSTM

In [ ]:
lstm_result = fit(
    lstm_model,
    X_train, y_train,
    X_val, y_val,
    **TRAINING_CONFIG,
)

lstm_model = lstm_result["model"]

print("Training time:", lstm_result["training_time"])
print("Epochs:", lstm_result["epochs_completed"])

# 19. Collect Training Results

In [ ]:
training_results = {
    "DNN": dnn_result,
    "RNN": rnn_result,
    "CNN": cnn_result,
    "LSTM": lstm_result,
}

trained_models = {
    name: result["model"]
    for name, result in training_results.items()
}

# 20. Loss Curves

In [ ]:
for name, result in training_results.items():
    history = result["history"]

    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="Training")
    plt.plot(history["val_loss"], label="Validation")
    plt.title(f"{name}: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

# 21. Accuracy Curves

In [ ]:
for name, result in training_results.items():
    history = result["history"]

    plt.figure(figsize=(8, 4))
    plt.plot(history["train_accuracy"], label="Training")
    plt.plot(history["val_accuracy"], label="Validation")
    plt.title(f"{name}: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

### Training behavior analysis

For each architecture discuss:

- convergence speed;
- overfitting;
- underfitting;
- stability;
- early stopping;
- train/validation gap.

**TODO:** Write your analysis here.

# 22. Final Test Evaluation

The test set should be used for final evaluation, not for hyperparameter selection.

Report:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Weighted F1
- Per-class metrics

In [ ]:
test_results = {}

for name, model in trained_models.items():
    test_results[name] = evaluate_model(
        model,
        X_test,
        y_test,
    )

comparison_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": result["accuracy"],
        "Macro Precision": result["macro_precision"],
        "Macro Recall": result["macro_recall"],
        "Macro F1": result["macro_f1"],
        "Weighted F1": result["weighted_f1"],
    }
    for name, result in test_results.items()
])

comparison_df

# 23. Per-Class Classification Reports

In [ ]:
for name, result in test_results.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(
        classification_report(
            y_test,
            result["predictions"],
            target_names=class_names,
            digits=4,
        )
    )

# 24. Confusion Matrices

In [ ]:
for name, result in test_results.items():
    cm = confusion_matrix(
        y_test,
        result["predictions"],
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    fig, ax = plt.subplots(figsize=(9, 8))
    disp.plot(ax=ax, xticks_rotation=45)
    plt.title(f"{name}: Confusion Matrix")
    plt.tight_layout()
    plt.show()

### Activity-specific error analysis

Pay particular attention to:

- WALKING vs WALKING_UPSTAIRS vs WALKING_DOWNSTAIRS;
- SITTING vs STANDING;
- whether LAYING is easier or harder;
- whether recurrent models make similar errors;
- whether CNN differs from RNN/LSTM;
- whether flattening disadvantages the DNN.

**TODO:** Write your analysis here.

# 25. Trainable Parameter Counts

In [ ]:
parameter_counts = {
    name: count_parameters(model)
    for name, model in trained_models.items()
}

parameter_df = pd.DataFrame({
    "Model": list(parameter_counts.keys()),
    "Parameters": list(parameter_counts.values()),
})

parameter_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(parameter_df["Model"], parameter_df["Parameters"])
plt.title("Trainable Parameters")
plt.xlabel("Model")
plt.ylabel("Number of Parameters")
plt.tight_layout()
plt.show()

# 26. Training-Time Comparison

In [ ]:
training_time_df = pd.DataFrame([
    {
        "Model": name,
        "Training Time": result["training_time"],
        "Epochs": result["epochs_completed"],
        "Average Time / Epoch": (
            result["training_time"] /
            max(result["epochs_completed"], 1)
        ),
    }
    for name, result in training_results.items()
])

training_time_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    training_time_df["Model"],
    training_time_df["Training Time"],
)
plt.title("Total Training Time")
plt.xlabel("Model")
plt.ylabel("Time")
plt.tight_layout()
plt.show()

# 27. Inference-Time Comparison

In [ ]:
inference_times = {
    name: measure_inference_time(model, X_test)
    for name, model in trained_models.items()
}

inference_df = pd.DataFrame({
    "Model": list(inference_times.keys()),
    "Inference Time": list(inference_times.values()),
})

inference_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    inference_df["Model"],
    inference_df["Inference Time"],
)
plt.title("Inference Time")
plt.xlabel("Model")
plt.ylabel("Time")
plt.tight_layout()
plt.show()

# 28. Complete Model Comparison

In [ ]:
final_comparison = comparison_df.copy()

final_comparison["Parameters"] = (
    final_comparison["Model"].map(parameter_counts)
)

final_comparison["Training Time"] = (
    final_comparison["Model"].map({
        name: result["training_time"]
        for name, result in training_results.items()
    })
)

final_comparison["Inference Time"] = (
    final_comparison["Model"].map(inference_times)
)

final_comparison

# 29. Predictive-Metric Comparison

In [ ]:
metric_columns = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
]

final_comparison.set_index("Model")[metric_columns].plot(
    kind="bar",
    figsize=(12, 6),
)

plt.title("Predictive Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 30. Performance vs Computational Cost

In [ ]:
plt.figure(figsize=(8, 6))

for _, row in final_comparison.iterrows():
    plt.scatter(
        row["Training Time"],
        row["Macro F1"],
        s=100,
    )
    plt.annotate(
        row["Model"],
        (row["Training Time"], row["Macro F1"]),
    )

plt.xlabel("Training Time")
plt.ylabel("Macro F1")
plt.title("Macro F1 vs Training Time")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for _, row in final_comparison.iterrows():
    plt.scatter(
        row["Parameters"],
        row["Macro F1"],
        s=100,
    )
    plt.annotate(
        row["Model"],
        (row["Parameters"], row["Macro F1"]),
    )

plt.xlabel("Trainable Parameters")
plt.ylabel("Macro F1")
plt.title("Macro F1 vs Model Size")
plt.tight_layout()
plt.show()

### Performance/cost discussion

Does increased architectural complexity produce a meaningful performance improvement relative to its computational cost?

**TODO:** Write your analysis here.

# 31. Repeated Experiments: Three Random Seeds

Run the main comparison with at least:

```python
SEEDS = [1, 42, 123]
```

Instantiate a **new model** for every run.

Do not reuse previously trained weights.

In [ ]:
SEEDS = [1, 42, 123]

seed_experiment_results = []

# TODO:
#
# for seed in SEEDS:
#     1. set_seed(seed)
#     2. instantiate fresh DNN/RNN/CNN/LSTM objects
#     3. train each model
#     4. evaluate it
#     5. record:
#        - Seed
#        - Model
#        - Accuracy
#        - Macro F1
#        - Weighted F1
#        - Training Time
#        - Parameters
#
# Append one dictionary per run to seed_experiment_results.

# YOUR CODE HERE

# 32. Aggregate Repeated Runs

In [ ]:
seed_df = pd.DataFrame(seed_experiment_results)
seed_df

In [ ]:
if not seed_df.empty:
    seed_summary = (
        seed_df
        .groupby("Model")
        .agg({
            "Accuracy": ["mean", "std"],
            "Macro F1": ["mean", "std"],
            "Weighted F1": ["mean", "std"],
            "Training Time": ["mean", "std"],
        })
    )

    display(seed_summary)
else:
    print("Complete the repeated-seed experiment first.")

### Stability analysis

Discuss:

- variability across seeds;
- whether architecture comparisons remain consistent;
- whether a conclusion based on only one run would be reliable.

**TODO:** Write your analysis here.

# 33. Controlled Hyperparameter Experiment

Perform at least one controlled experiment.

Examples:

- DNN hidden dimension or dropout;
- RNN hidden size;
- CNN kernel size;
- CNN filter count;
- LSTM hidden size;
- batch size;
- learning rate.

Change one factor systematically and keep other important conditions fixed.

In [ ]:
experiment_values = [32, 64, 128, 256]
controlled_experiment_results = []

for value in experiment_values:
    # TODO:
    # - instantiate a fresh selected model
    # - use `value` for the selected hyperparameter
    # - train
    # - evaluate
    # - record predictive metrics, time and parameters
    pass

controlled_df = pd.DataFrame(controlled_experiment_results)
controlled_df

### Controlled-experiment discussion

Explain:

- the parameter selected;
- why it was selected;
- effect on predictive performance;
- effect on computational cost;
- whether larger/more complex settings consistently improved generalization.

**TODO:** Write your discussion here.

# 34. DNN Analysis

Discuss:

- consequences of flattening the 128 × 9 sequence;
- predictive performance;
- parameter count;
- convergence;
- activity-specific errors;
- computational cost.

**TODO:** Write your analysis here.

# 35. RNN Analysis

Discuss:

- benefit of explicit sequential processing;
- convergence behavior;
- activity-specific errors;
- training/inference cost;
- parameter count;
- potential limitations of vanilla recurrence.

**TODO:** Write your analysis here.

# 36. CNN Analysis

Discuss:

- ability to detect local temporal sensor patterns;
- performance on dynamic activities;
- computational efficiency;
- effect of convolution/filter design;
- parameter count.

**TODO:** Write your analysis here.

# 37. LSTM Analysis

Discuss:

- whether gating improves sequence classification;
- performance relative to vanilla RNN;
- computational overhead;
- activity-specific improvements/errors;
- convergence and stability.

**TODO:** Write your analysis here.

# 38. RNN vs LSTM

Directly compare the two recurrent architectures in terms of:

- Accuracy
- Macro F1
- per-class performance
- convergence
- training time
- inference time
- trainable parameters
- stability across seeds

Does the LSTM's additional gating complexity provide a measurable benefit on this dataset?

Support your answer with your experimental evidence.

**TODO:** Write your analysis here.

# 39. Final Comparative Discussion

Synthesize the complete project.

Do **not** simply state which model obtained the highest accuracy.

Discuss:

1. **Predictive performance:** How do DNN, RNN, CNN and LSTM differ?
2. **Activity-specific performance:** Which activities are difficult and why?
3. **Temporal structure:** Which architectures appear to exploit the sensor sequences effectively?
4. **Computational efficiency:** Compare training time, inference time and parameter count.
5. **Generalization:** Which architectures overfit or underfit?
6. **Stability:** Are results consistent across seeds?
7. **Complexity:** Does a more sophisticated architecture necessarily perform better?
8. **Trade-offs:** Under what conditions would each architecture be reasonable?
9. **Human activity recognition:** What do the results suggest about modeling smartphone inertial signals?

Every important conclusion must be supported by your experimental results.

**TODO:** Write your final discussion here.

# 40. Save Experimental Results

In [ ]:
metrics_dir = PROJECT_ROOT / "results" / "metrics"
figures_dir = PROJECT_ROOT / "results" / "figures"
models_dir = PROJECT_ROOT / "results" / "models"

metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

final_comparison.to_csv(
    metrics_dir / "model_comparison.csv",
    index=False,
)

if "seed_df" in globals() and not seed_df.empty:
    seed_df.to_csv(
        metrics_dir / "seed_results.csv",
        index=False,
    )

if "controlled_df" in globals() and not controlled_df.empty:
    controlled_df.to_csv(
        metrics_dir / "controlled_experiment.csv",
        index=False,
    )

print("Saved results to:", metrics_dir)

# 41. Reproducibility Information

In [ ]:
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 42. Final Submission Checklist

## Files

- [ ] `README.md`
- [ ] `requirements.txt`
- [ ] instructor-provided `prepare_dataset.py`
- [ ] prepared `.npz` dataset
- [ ] `metadata.json`
- [ ] `src/models.py`
- [ ] `src/training.py`
- [ ] `src/evaluation.py`
- [ ] `src/utils.py`
- [ ] tests
- [ ] executed notebook
- [ ] saved metrics
- [ ] saved figures

## Models

- [ ] DNN
- [ ] RNN
- [ ] CNN
- [ ] LSTM

## Analysis

- [ ] dataset exploration
- [ ] sensor-sequence visualization
- [ ] class distribution
- [ ] loss curves
- [ ] accuracy curves
- [ ] Accuracy
- [ ] Macro Precision
- [ ] Macro Recall
- [ ] Macro F1
- [ ] Weighted F1
- [ ] per-class metrics
- [ ] four confusion matrices
- [ ] trainable parameter comparison
- [ ] training-time comparison
- [ ] inference-time comparison
- [ ] at least three random seeds
- [ ] mean and standard deviation
- [ ] controlled hyperparameter experiment
- [ ] performance/cost analysis
- [ ] RNN vs LSTM analysis
- [ ] final comparative discussion

## Academic integrity

- [ ] I wrote and understand the code I submitted.
- [ ] I can explain and modify my model implementations.
- [ ] I can explain my experimental results.
- [ ] I did not use prohibited generative-AI/LLM assistance to generate the submitted project code, experiments, analysis, or written answers.

**Prohibited generative-AI/LLM-generated work is subject to the course academic-integrity policy. If prohibited use is established, the affected project code will not be evaluated and no credit will be awarded for that work, subject to institutional procedures.**

## Submission

Submit the complete project as:

```text
dlfbt_final_project.zip
```

The notebook must be **already executed**, with outputs, figures, tables and written discussions visible.